# 🎙️ OmniVoice — English / Cantonese / Mixed TTS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kychugo/OmniVoice/blob/master/docs/OmniVoice_Cantonese_English.ipynb)

This notebook lets you generate speech in **English**, **Cantonese (廣東話)**, or a **mixed** style using [OmniVoice](https://github.com/k2-fsa/OmniVoice).

---

## 📖 How to Use This Notebook

Follow the steps **from top to bottom** — each section builds on the previous one:

| Step | Cell | What it does |
|------|------|--------------|
| 1 | **Install** | Installs OmniVoice (run once per session). |
| 2 | **Load model** | Downloads and loads the pre-trained model onto the GPU. Takes ~1–2 min on first run. |
| 3 | **Select language** | Choose English, Cantonese, or Mixed. |
| 4 | **Enter your text** | Type the text you want spoken. |
| 5 | **Voice options** *(optional)* | Customise gender, pitch, accent/dialect, speed, etc. Press Enter to skip any field. |
| 6 | **Generate audio** | Runs inference and plays the audio right inside Colab. |
| 7 | **Download** | Downloads the generated `.wav` file to your computer. |

### 💡 Tips

- **Runtime → Change runtime type → T4 GPU** for fast inference.
- You only need to run Steps 1 and 2 **once** per Colab session. Steps 3–7 can be re-run as many times as you like to generate different audio.
- All optional fields in Step 5 can be **left blank** (just press Enter / leave the field empty).
- For **Voice Cloning** (using your own voice as reference), see the [official OmniVoice notebook](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb).

### 🗣️ Language hints

| Language | Example text |
|----------|--------------|
| English | `Hello, how are you today?` |
| Cantonese | `你好，今日天氣點呀？` |
| Mixed (code-switch) | `Hello！今日係 Monday，要返工喇。` |

---
## Step 1 — Install OmniVoice

> Run this cell once. Colab already has PyTorch + CUDA — we only need to install OmniVoice.

In [ ]:
!pip install -q omnivoice

---
## Step 2 — Load the Model

> This downloads the model weights (~2 GB) and loads them onto the GPU. Run once per session.

In [ ]:
import torch
import soundfile as sf
from IPython.display import Audio, display
from omnivoice import OmniVoice

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading OmniVoice on {device} ({dtype}) …")
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=device,
    dtype=dtype,
)
print("✅ Model loaded!")

---
## Step 3 — Select Language

Run the cell below and choose your language from the dropdown.

| Choice | Meaning |
|--------|---------|
| `English` | Pure English speech |
| `Cantonese` | Pure Cantonese (廣東話) speech |
| `Mixed` | Code-switching — you can mix English and Cantonese in the same sentence |

In [ ]:
# ── Language selection ────────────────────────────────────────────────────────
# language_id: 'en' = English, 'yue' = Cantonese, None = auto-detect (Mixed)

import ipywidgets as widgets
from IPython.display import display as _display

lang_dropdown = widgets.Dropdown(
    options=[
        ("English", "en"),
        ("Cantonese (廣東話)", "yue"),
        ("Mixed (English + Cantonese)", None),
    ],
    value="en",
    description="Language:",
    style={"description_width": "initial"},
)
_display(lang_dropdown)
print("Select your language above, then run the next cells.")

---
## Step 4 — Enter Your Text

Type the text you want converted to speech in the text box below.

In [ ]:
text_box = widgets.Textarea(
    value="",
    placeholder="Type your text here… (English, Cantonese, or mixed)",
    description="Text:",
    layout=widgets.Layout(width="80%", height="100px"),
    style={"description_width": "initial"},
)
_display(text_box)
print("Enter your text above, then proceed to Step 5.")

---
## Step 5 — Voice Options *(all optional — leave blank to skip)*

Customise the generated voice. **Every field is optional** — just leave it blank to use the default.

### Voice Design attributes (comma-separated, e.g. `female, young adult, high pitch`)

| Category | English options | Chinese options |
|----------|-----------------|-----------------|
| **Gender** | `male` / `female` | `男` / `女` |
| **Age** | `child`, `teenager`, `young adult`, `middle-aged`, `elderly` | `儿童`, `少年`, `青年`, `中年`, `老年` |
| **Pitch** | `very low pitch`, `low pitch`, `moderate pitch`, `high pitch`, `very high pitch` | `极低音调` … `极高音调` |
| **Style** | `whisper` | `耳语` |
| **English accent** | `american accent`, `british accent`, `australian accent`, `canadian accent`, `indian accent`, `chinese accent` | |
| **Cantonese dialect** | — | `四川话`, `东北话`, `河南话`, … |

You can freely combine attributes across categories: e.g. `female, young adult, high pitch, british accent`.

In [ ]:
instruct_box = widgets.Text(
    value="",
    placeholder="e.g. female, young adult, high pitch  (leave blank to skip)",
    description="Voice design:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "initial"},
)

speed_box = widgets.FloatSlider(
    value=1.0, min=0.5, max=2.0, step=0.1,
    description="Speed:",
    readout_format=".1f",
    layout=widgets.Layout(width="60%"),
    style={"description_width": "initial"},
)
speed_label = widgets.HTML(
    "<small>&nbsp;1.0 = normal &nbsp;|&nbsp; &gt;1.0 = faster &nbsp;|&nbsp; &lt;1.0 = slower</small>"
)

steps_box = widgets.IntSlider(
    value=32, min=8, max=64, step=8,
    description="Diffusion steps:",
    layout=widgets.Layout(width="60%"),
    style={"description_width": "initial"},
)
steps_label = widgets.HTML(
    "<small>&nbsp;16 = faster / lower quality &nbsp;|&nbsp; 32 = default &nbsp;|&nbsp; 64 = highest quality</small>"
)

_display(
    instruct_box,
    widgets.HBox([speed_box, speed_label]),
    widgets.HBox([steps_box, steps_label]),
)
print("Adjust the options above (or leave defaults), then run Step 6 to generate audio.")

---
## Step 6 — Generate Audio 🎧

Runs inference using your text and options. The audio will play directly in this cell.

In [ ]:
# ── Collect inputs ────────────────────────────────────────────────────────────
selected_language = lang_dropdown.value          # 'en', 'yue', or None
selected_text     = text_box.value.strip()
try:
    selected_instruct = instruct_box.value.strip()
    selected_speed    = float(speed_box.value)
    selected_steps    = int(steps_box.value)
except NameError:
    selected_instruct = ""
    selected_speed    = 1.0
    selected_steps    = 32
    print("⚠️  Step 5 widgets not found — using default voice options (instruct=none, speed=1.0, steps=32).")

# ── Validate ──────────────────────────────────────────────────────────────────
if not selected_text:
    raise ValueError("❌ Text is empty. Please enter some text in Step 4 first.")

lang_name_map = {"en": "English", "yue": "Cantonese (廣東話)", None: "Mixed / Auto-detect"}
print(f"Language   : {lang_name_map[selected_language]}")
print(f"Text       : {selected_text}")
print(f"Voice design: {selected_instruct or '(none — auto voice)'}")
print(f"Speed      : {selected_speed}x")
print(f"Steps      : {selected_steps}")
print()

# ── Build generate() kwargs ───────────────────────────────────────────────────
gen_kwargs = dict(
    text=selected_text,
    num_step=selected_steps,
    speed=selected_speed if selected_speed != 1.0 else None,
)

if selected_instruct:
    gen_kwargs["instruct"] = selected_instruct

if selected_language is not None:
    gen_kwargs["language"] = selected_language

# ── Run inference ─────────────────────────────────────────────────────────────
print("⏳ Generating audio …")
audio_list = model.generate(**gen_kwargs)
audio_np   = audio_list[0]   # numpy array, shape (T,), sample rate 24 kHz

OUTPUT_FILE = "omnivoice_output.wav"
sf.write(OUTPUT_FILE, audio_np, 24000)
print(f"✅ Audio saved to '{OUTPUT_FILE}'")
print()

# ── Play inline ───────────────────────────────────────────────────────────────
display(Audio(audio_np, rate=24000, autoplay=True))

---
## Step 7 — Download the Audio File

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)
print(f"⬇️  Downloading '{OUTPUT_FILE}' …")

---
## ♻️ Generate Again

To generate another audio clip:
1. Go back to **Step 3** (or whichever step you want to change).
2. Update your selections.
3. Re-run **Step 6** (and **Step 7** to download).

You do **not** need to re-run Steps 1 or 2 — the model stays loaded in memory for the whole session.

---

## 🔗 Resources

- 📄 [OmniVoice Paper (arXiv)](https://arxiv.org/abs/2604.00688)
- 🤗 [Model on Hugging Face](https://huggingface.co/k2-fsa/OmniVoice)
- 🌐 [Demo Space on Hugging Face](https://huggingface.co/spaces/k2-fsa/OmniVoice)
- 📝 [Voice Design attributes reference](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md)
- ⚙️ [Generation parameters reference](https://github.com/k2-fsa/OmniVoice/blob/master/docs/generation-parameters.md)
- 🌍 [Full language list (600+ languages)](https://github.com/k2-fsa/OmniVoice/blob/master/docs/languages.md)